23070521054 - GAURI GULHANE

In [3]:
import urllib.request
from pathlib import Path
import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# Ensure the credit card dataset exists. If not found locally, we can download it or prompt.
data_path = Path("creditcard.csv")
if not data_path.exists():
    print("Downloading creditcard.csv...")
    # Using a verified raw git repository for the Credit Card Fraud Detection dataset
    url = "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
    urllib.request.urlretrieve(url, data_path)

# Load and split dataset
data = pd.read_csv(data_path)
X = data.drop(columns="Class")
y = data["Class"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features using only the training data to avoid data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Apply SMOTE to training data to handle class imbalance
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print(f"Dataset shape: {data.shape}")
print(f"Fraud transactions before SMOTE: {y.sum()} ({y.mean() * 100:.2f}%)")
print(f"Training class counts after SMOTE: {dict(pd.Series(y_train).value_counts())}")

Dataset shape: (284807, 31)
Fraud transactions before SMOTE: 492 (0.17%)
Training class counts after SMOTE: {0: np.int64(227451), 1: np.int64(227451)}


In [4]:
# Build the Deep Neural Network model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=2048,
    verbose=1
)

# Evaluate the model
probabilities = model.predict(X_test, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["Genuine", "Fraud"], zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, predictions))

Epoch 1/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8937 - loss: 0.2720 - val_accuracy: 0.9123 - val_loss: 0.1669
Epoch 2/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9626 - loss: 0.1021 - val_accuracy: 0.9354 - val_loss: 0.1274
Epoch 3/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9704 - loss: 0.0770 - val_accuracy: 0.9555 - val_loss: 0.0973
Epoch 4/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9763 - loss: 0.0625 - val_accuracy: 0.9684 - val_loss: 0.0759
Epoch 5/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9805 - loss: 0.0526 - val_accuracy: 0.9783 - val_loss: 0.0595
Epoch 6/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9842 - loss: 0.0441 - val_accuracy: 0.9820 - val_loss: 0.0513
Epoch 7/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9867 - loss: 0.0379 - val_accuracy: 0.9893 - val_loss: 0.0391
Epoch 8/10
178/178 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9887 - loss: 0.0328 - val_accuracy: 0.